In [ ]:
# Реализуйте пакетный градиентный спуск (**Batch Gradient Descent**) с ранней остановкой (**Early Stopping**) 
# для Softmax Regression без использования Scikit-Learn, только с помощью NumPy. 
# Используйте его для задачи классификации, например, на датасете Iris.

In [1]:
# Detect iris virginica
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
print(list(iris), "\n")
print(iris.data.head(3),  "\n")
# instances are not shuffled
print(iris.target.head(3),  "\n")
print(iris.target_names,  "\n")
# чашелистик, лепесток

['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'] 

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                4.9               3.0                1.4               0.2
2                4.7               3.2                1.3               0.2 

0    0
1    0
2    0
Name: target, dtype: int32 

['setosa' 'versicolor' 'virginica'] 



In [27]:
# Detect iris virginica
import numpy as np
import matplotlib.pyplot as plt

X = iris.data[["petal width (cm)"]].values
y = iris.target_names[iris.target] == "virginica"
# split data 80:20
def split_data(x, y):
    # X_train, X_test, y_train, y_test
    train_end = int(len(X) / 100 * 80)
    valid_start = train_end+1
    X_train = x[0:train_end]
    y_train = x[0:train_end]
    X_test = x[valid_start:len(X)]
    y_test = y[valid_start:len(X)]
    return X_train,y_train, X_test, y_test

X_train, y_train, X_test, y_test = split_data(X,y)
print(len(X_train))
print(len(y_train))
print(len(X_test))
print(len(y_test))

virginica_value_test = X[y][:1][0]
not_virginica_value_test = X[~y][:1][0]
print("Example with petal width is virginica",virginica_value_test)
print("Example with petal width is not virginica",not_virginica_value_test)


120
120
29
29
Example with petal width is virginica [2.5]
Example with petal width is not virginica [0.2]


In [ ]:
# Batch gradient
# Early stopping
# SoftmaxRegression

import math
import numpy as np


def generateXvector(X):
    """
    функция добавляет единицы к данным, чтобы можно было вычислять прогноз одним матричным умножением.
    """
    # np.c_ склеивает массивы по столбцам.
    # np.ones Добавляем столбец единиц слева от X
    vectorX = np.c_[np.ones((len(X), 1)), X]
    return vectorX


def theta_init(X):
    # Independent variables (X) — площадь и количество комнат. Это то, что мы подаём на вход модели.
    # Dependent variable (y) — цена. Это то, что мы хотим предсказать, и она зависит от входных признаков.
    # Количество элементов в theta = количество признаков + 1 (для свободного члена theta).
    # X = [ [50, 2], [2,3], [4,4]    # 2 признака]
    theta = np.random.randn(len(X[0]) + 1, 1)
    return theta


def sigmoid_function(X):
    return 1 / (1 + np.exp(-X))


class Logistics_Regression:
    learningrate = 0
    iterations = 0
    minError = None
    bestTheta = None
    def __init__(self, learningrate, iterations):
        self.learningrate = learningrate
        self.iterations = iterations

    def fit(self, X, y):
        y_new = np.reshape(y, (len(y), 1))
        cost_lst = []
        vectorX = generateXvector(X)
        theta = theta_init(X)
        m = len(X)
        for i in range(self.iterations):
            gradients = 2 / m * vectorX.T @ (sigmoid_function(vectorX @ (theta)) - y_new)
            theta = theta - self.learningrate  * gradients
            y_pred = sigmoid_function(vectorX @ theta)
            cost_value = -np.sum(y_new.T @ np.log(y_pred) + (1 - y_new).T @ np.log(1 - y_pred)) / len(y_pred)
            # Calculate the loss for each training instance
            cost_lst.append(cost_value)

            # Init values
            if self.minError is None or cost_value < self.minError:
                self.minError = cost_value
                self.bestTheta = theta

    def predict(self,val):
      vectorX = generateXvector(val)
      return sigmoid_function(vectorX @ self.bestTheta)

classifier = Logistics_Regression(0.2, 100)
classifier.fit(X,y)
print(classifier.minError, classifier.bestTheta,"\n")


print("Is virginica?",classifier.predict(virginica_value_test) > 0.5)
print("Is not?",classifier.predict(not_virginica_value_test) > 0.5)



0.3220029583496553 [[-2.97258792]
 [ 1.85814425]] 

[[ True]]
[[False]]
